# Demo 2 (new way) — Scalable processing in a Unified Studio notebook

Same Spark pipeline as the EMR demo, but run **interactively from a managed
Unified Studio project notebook**. No application to create, no submit call, no
`--py-files` plumbing, no S3 log-digging — the project gives you compute and the
code lives right here, governed and Git-backed.

**Run this in:** SageMaker Unified Studio → your project → JupyterLab, attached to
a Spark connection (Glue interactive session or EMR Serverless). The `spark`
session is provided by that connection.

_Cross-reference: implements the same [PIPELINE_SPEC](../../PIPELINE_SPEC.md) as
the EMR, Visual ETL, and Data Wrangler demos._

In [ ]:
# Config — the same raw locations every demo reads.
BUCKET = "roi-smdemo-029331796573-us-east-2"
TRIPS_PATH = f"s3://{BUCKET}/raw/trips/"
ZONES_PATH = f"s3://{BUCKET}/raw/zones/taxi_zone_lookup.csv"
OUTPUT_PATH = f"s3://{BUCKET}/processed/notebook/"

In [ ]:
# Reuse the SHARED pipeline (single source of truth) instead of re-typing it.
# addPyFile ships the module to the Spark workers, then we import it — proving the
# exact same code the EMR job ran also runs here.
spark.sparkContext.addPyFile(f"s3://{BUCKET}/code/taxi_transforms.py")
from taxi_transforms import build_pipeline, add_ml_label

In [ ]:
# Build the analytics-ready frame and peek — interactive, cell by cell.
df = build_pipeline(spark, TRIPS_PATH, ZONES_PATH)
print("rows:", df.count())
df.show(5, truncate=False)

In [ ]:
# A quick analytic you can't easily do mid-flow in Data Wrangler: average tip %
# by pickup borough. This is the payoff of running real Spark interactively.
from pyspark.sql import functions as F
(df.groupBy("pickup_borough")
   .agg(F.round(F.avg("tip_pct"), 2).alias("avg_tip_pct"),
        F.count("*").alias("trips"))
   .orderBy(F.desc("avg_tip_pct"))
   .show())

In [ ]:
# Write the result — same output contract as the EMR demo.
(df.write.mode("overwrite").partitionBy("pickup_date").parquet(OUTPUT_PATH))
print("wrote", OUTPUT_PATH)

## Old vs new — talking points

| | EMR Spark (old) | This notebook (new) |
|---|---|---|
| Get compute | create app + role + submit | pick a project connection |
| Ship code | `--py-files` to S3 | files live in the project |
| Iterate | submit → wait → read S3 logs | run a cell |
| Govern | none built-in | catalog, lineage, permissions, Git |

Same engine, same logic, dramatically less plumbing. That's the Unified Studio
pitch for data processing.